# AquaInsight — Task 6: Categorical Data Transformation

**Internship Project:** Predicting Water Quality Index for Comprehensive Water Assessment

This notebook is one of the nine independent GitHub deliverables. It can be run separately using the supplied water-quality CSV.

## Step-by-step approach

1. Load the supplied dataset.
2. Perform the task-specific analysis.
3. Display quantitative results.
4. Interpret the results for downstream water-quality modeling.
5. Preserve data for review rather than making unsupported automatic corrections.

In [ ]:
# Common setup — AquaInsight Water Quality Internship

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

try:
    from rapidfuzz.fuzz import ratio, token_set_ratio
except ImportError:
    raise ImportError("Install RapidFuzz first: pip install rapidfuzz")

possible_paths = [
    Path("Water Quality(1).csv"),
    Path("Water Quality.csv"),
    Path("../data/Water Quality(1).csv"),
    Path("../data/Water Quality.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the supplied CSV beside this notebook.")

df = pd.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head())


# Task 6 — Categorical Data Transformation

### Internship requirement
Identify categorical features, assess their cardinality, and compare appropriate encoding methods.

### Steps
1. Identify categorical variables.
2. Inspect their cardinality.
3. Apply one-hot encoding to nominal variables.
4. Demonstrate ordinal encoding only where an order is meaningful.
5. Apply frequency encoding to high-cardinality variables.
6. Demonstrate target encoding using training data only to avoid leakage.
7. Review the resulting feature dimensionality.


In [13]:
cat_demo = df[[
    "MonitoringLocationType",
    "ActivityType",
    "ResultUnit",
    "ResultStatusID",
    "CharacteristicName"
]].copy()

# Reduce very high-cardinality characteristic categories for a readable demo
cat_demo["CharacteristicGroup"] = np.where(
    cat_demo["CharacteristicName"].isin(df["CharacteristicName"].value_counts().head(10).index),
    cat_demo["CharacteristicName"],
    "Other"
)

onehot = pd.get_dummies(
    cat_demo[["MonitoringLocationType", "ActivityType", "ResultUnit", "CharacteristicGroup"]],
    drop_first=True
)

frequency_maps = {
    col: cat_demo[col].value_counts(normalize=True)
    for col in ["MonitoringLocationType", "ActivityType", "ResultUnit", "CharacteristicGroup"]
}

frequency_encoded = cat_demo.copy()
for col, mapping in frequency_maps.items():
    frequency_encoded[col + "_freq"] = frequency_encoded[col].map(mapping)

display(onehot.head())
display(frequency_encoded.head())

# Ordinal encoding is demonstrated only for variables where an order is meaningful.
ordinal = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
ordinal_demo = ordinal.fit_transform(
    cat_demo[["MonitoringLocationType", "ActivityType"]]
)
display(pd.DataFrame(ordinal_demo, columns=["MonitoringLocationType_ordinal", "ActivityType_ordinal"]).head())

print("For model training, high-cardinality categories such as CharacteristicName should be handled carefully to avoid a huge sparse feature matrix.")


,MonitoringLocationType_River/Stream,ActivityType_Sample-Routine,ResultUnit_%,ResultUnit_JTU,ResultUnit_NTU,ResultUnit_TCU,ResultUnit_deg C,ResultUnit_m,ResultUnit_mg/L,ResultUnit_ng/L,...,CharacteristicGroup_Calcium,CharacteristicGroup_Chloride,CharacteristicGroup_Magnesium,CharacteristicGroup_Organic carbon,CharacteristicGroup_Other,CharacteristicGroup_Sodium,CharacteristicGroup_Specific conductance,CharacteristicGroup_Sulfate,"CharacteristicGroup_Total Nitrogen, mixed forms",CharacteristicGroup_pH
0,True,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,True,True,False,False,False,True,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2,True,True,False,False,False,False,False,False,True,False,...,True,False,False,False,False,False,False,False,False,False
3,True,True,False,False,False,False,False,False,True,False,...,False,True,False,False,False,False,False,False,False,False
4,True,True,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False


,MonitoringLocationType,ActivityType,ResultUnit,ResultStatusID,CharacteristicName,CharacteristicGroup,MonitoringLocationType_freq,ActivityType_freq,ResultUnit_freq,CharacteristicGroup_freq
0,River/Stream,Sample-Routine,ug/L,Preliminary,Aluminum,Aluminum,0.775401,0.955536,0.500792,0.030672
1,River/Stream,Sample-Routine,TCU,Preliminary,Apparent color,Other,0.775401,0.955536,0.031470,0.669553
2,River/Stream,Sample-Routine,mg/L,Preliminary,Calcium,Calcium,0.775401,0.955536,0.381500,0.031781
3,River/Stream,Sample-Routine,mg/L,Preliminary,Chloride,Chloride,0.775401,0.955536,0.381500,0.031533
4,River/Stream,Sample-Routine,mg/L,Preliminary,Gran acid neutralizing capacity,Other,0.775401,0.955536,0.381500,0.669553


,MonitoringLocationType_ordinal,ActivityType_ordinal
0,1.0,1.0
1,1.0,1.0
2,1.0,1.0
3,1.0,1.0
4,1.0,1.0


For model training, high-cardinality categories such as CharacteristicName should be handled carefully to avoid a huge sparse feature matrix.


In [14]:
# Leakage-safe target encoding demonstration
# Here ResultValue is only used as an illustrative continuous target.
# In a real WQI model, replace it with the official WQI target.
te_df = df[["CharacteristicName", "ResultValue"]].dropna().sample(
    min(30000, df["ResultValue"].notna().sum()), random_state=42
)

split = int(len(te_df) * 0.8)
train_te = te_df.iloc[:split].copy()
valid_te = te_df.iloc[split:].copy()

global_mean = train_te["ResultValue"].mean()
stats_te = train_te.groupby("CharacteristicName")["ResultValue"].agg(["mean", "count"])
smoothing = 20
stats_te["smoothed_mean"] = (
    (stats_te["count"] * stats_te["mean"] + smoothing * global_mean)
    / (stats_te["count"] + smoothing)
)

train_te["Characteristic_target_encoded"] = (
    train_te["CharacteristicName"].map(stats_te["smoothed_mean"]).fillna(global_mean)
)
valid_te["Characteristic_target_encoded"] = (
    valid_te["CharacteristicName"].map(stats_te["smoothed_mean"]).fillna(global_mean)
)

display(train_te.head())
display(valid_te.head())
print("Target encoding is calculated from training data only, preventing validation leakage.")


,CharacteristicName,ResultValue,Characteristic_target_encoded
139766,Iron,0.850,90.108009
156707,Strontium,72.000,17.976295
17995,Titanium,2.500,2.276837
215937,Thallium,0.011,5.939702
38892,Specific conductance,23.900,85.557888


,CharacteristicName,ResultValue,Characteristic_target_encoded
108172,Organic carbon,9.40,10.148483
164678,pH,4.95,6.304572
159144,Turbidity,1.30,4.627638
44538,Nickel,0.20,1.254021
39189,Arsenic,1.50,1.152972


Target encoding is calculated from training data only, preventing validation leakage.


## Task 6 — Conclusion

The analysis above completes the requested **Categorical Data Transformation** component of the AquaInsight internship assignment. Results should be interpreted together with domain requirements and the official WQI definition when it becomes available.